# Retail Customer Insights

## RFM segmentation and cohort analysis

**A reproducible RFM segmentation of the Online Retail transaction dataset.** This notebook turns transaction-line data into customer-level recency, purchase frequency, and net sales measures, then translates those measures into segments and campaign hypotheses.

> This is a descriptive, historical analysis. Segment revenue is past net sales, not forecast revenue or guaranteed recoverable revenue.

## 1. Setup and data loading

Place `online_retail_final.csv` beside this notebook. Dates are parsed day-first because the source uses `DD-MM-YYYY HH:MM`. The code below checks required fields and derives sales from quantity and unit price to avoid relying on a precomputed total.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = Path('online_retail_final.csv')
REQUIRED = {'InvoiceNo','Quantity','InvoiceDate','UnitPrice','CustomerID'}
raw = pd.read_csv(DATA_PATH)
missing = REQUIRED - set(raw.columns)
if missing:
    raise ValueError(f'Missing required columns: {sorted(missing)}')
raw['InvoiceDate'] = pd.to_datetime(raw['InvoiceDate'], dayfirst=True, errors='coerce')
raw['CustomerID'] = pd.to_numeric(raw['CustomerID'], errors='coerce')
raw['Quantity'] = pd.to_numeric(raw['Quantity'], errors='coerce')
raw['UnitPrice'] = pd.to_numeric(raw['UnitPrice'], errors='coerce')
raw['LineSales'] = raw['Quantity'] * raw['UnitPrice']
print(f'{len(raw):,} source rows; {raw.InvoiceDate.min()} to {raw.InvoiceDate.max()}')
raw.head()

## 2. Data quality and purchase definition

For this analysis, a purchase is a positive-quantity, positive-price line with a customer ID and valid date. Credit notes, returns, zero-value lines, and unidentified customers are excluded from customer purchase RFM. This means Monetary is **gross positive sales**; returns are excluded rather than netted against purchases. Review the audit counts before interpreting results.

In [ ]:
audit = pd.Series({
    'source rows': len(raw),
    'missing customer': raw.CustomerID.isna().sum(),
    'invalid date': raw.InvoiceDate.isna().sum(),
    'nonpositive quantity': (raw.Quantity <= 0).sum(),
    'nonpositive unit price': (raw.UnitPrice <= 0).sum(),
    'cancellation invoice rows': raw.InvoiceNo.astype(str).str.startswith('C', na=False).sum(),
})
audit

In [ ]:
valid = (raw.CustomerID.notna() & raw.InvoiceDate.notna() & raw.Quantity.gt(0) & raw.UnitPrice.gt(0) & ~raw.InvoiceNo.astype(str).str.startswith('C', na=False))
sales = raw.loc[valid].copy()
if sales.empty:
    raise ValueError('No valid purchase lines remain after filtering.')
print(f'{len(sales):,} qualifying lines from {sales.CustomerID.nunique():,} customers')
print(f'Gross positive sales: £{sales.LineSales.sum():,.2f}')

## 3. Build customer-level RFM

Frequency is the number of distinct qualifying invoices, not line items. Recency is days since the latest qualifying purchase, measured from one day after the last observed transaction. Monetary is the sum of positive line sales. These definitions make the measures comparable and auditable.

In [ ]:
analysis_date = sales.InvoiceDate.max().normalize() + pd.Timedelta(days=1)
rfm = sales.groupby('CustomerID').agg(
    LastPurchase=('InvoiceDate','max'),
    Frequency=('InvoiceNo','nunique'),
    Monetary=('LineSales','sum'),
).copy()
rfm['Recency'] = (analysis_date - rfm.LastPurchase.dt.normalize()).dt.days
rfm = rfm[['Recency','Frequency','Monetary']]
assert rfm.index.is_unique and rfm.notna().all().all()
rfm.describe().round(2)

## 4. Score and assign segments

Quintiles provide relative scores within this dataset. Recency is reversed so 5 means most recent; frequency and monetary score 5 means higher activity/value. Ties can prevent equal-sized bins, so `duplicates="drop"` is used and the resulting score ranges are reported. Segment names are assigned from Recency and Frequency only; Monetary remains available for profiling instead of silently affecting the label.

In [ ]:
def quintile_score(series, reverse=False):
    # Rank ties deterministically so qcut can form five populated groups.
    ranked = series.rank(method='first')
    q = pd.qcut(ranked, q=5, labels=False, duplicates='drop') + 1
    score = q.astype(int)
    return (score.max() + 1 - score) if reverse else score

rfm['R'] = quintile_score(rfm.Recency, reverse=True)
rfm['F'] = quintile_score(rfm.Frequency)
rfm['M'] = quintile_score(rfm.Monetary)
rfm['RFM_Score'] = rfm[['R','F','M']].sum(axis=1)
rfm[['Recency','Frequency','Monetary','R','F','M','RFM_Score']].head()

In [ ]:
def segment_customer(row):
    r, f = row.R, row.F
    if r >= 4 and f >= 4: return 'Champions'
    if r >= 3 and f >= 4: return 'Loyal customers'
    if r >= 4 and f >= 2: return 'Potential loyalists'
    if r == 5 and f == 1: return 'New customers'
    if r >= 3 and f == 1: return 'Promising'
    if r == 3 and f <= 3: return 'Needs attention'
    if r <= 2 and f >= 4: return 'Cannot lose them'
    if r <= 2 and f >= 2: return 'At risk'
    return 'Hibernating'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)
assert rfm.Segment.notna().all()
rfm.Segment.value_counts()

## 5. Segment profile

Customer count, average behavior, historic sales, and share of sales are reported together. A large historic sales share identifies where past value sits; it does not estimate future loss or campaign uplift.

In [ ]:
summary = rfm.groupby('Segment').agg(
    Customers=('Segment','size'),
    AvgRecencyDays=('Recency','mean'),
    AvgInvoices=('Frequency','mean'),
    AvgHistoricSales=('Monetary','mean'),
    HistoricSales=('Monetary','sum'),
).sort_values('HistoricSales', ascending=False)
summary['CustomerSharePct'] = 100 * summary.Customers / len(rfm)
summary['HistoricSalesSharePct'] = 100 * summary.HistoricSales / rfm.Monetary.sum()
summary.round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
order = summary.index
sns.barplot(data=summary.reset_index(), y='Segment', x='Customers', order=order, ax=axes[0], color='#3572A5')
axes[0].set_title('Customers by segment'); axes[0].set_xlabel('Customers')
sns.barplot(data=summary.reset_index(), y='Segment', x='HistoricSalesSharePct', order=order, ax=axes[1], color='#E07A3F')
axes[1].set_title('Share of historic sales'); axes[1].set_xlabel('Sales share (%)')
plt.tight_layout(); plt.show()

## 6. Extra decision views

These views add context beyond the segment labels: repeat purchase behavior, cohort retention, and geography. Cohorts are based on each customer’s first qualifying purchase month; the table reports the share of that cohort active in later calendar months. It is descriptive and is not a forecast.


In [ ]:
# Repeat-purchase and concentration indicators
customer_orders = sales.groupby('CustomerID')['InvoiceNo'].nunique()
print(f"Repeat-purchase customers: {(customer_orders >= 2).mean():.1%}")
print(f"Median invoices per customer: {customer_orders.median():.0f}")
print(f"Top 10% customer share of gross sales: {rfm.Monetary.nlargest(max(1, int(np.ceil(len(rfm)*.10)))).sum()/rfm.Monetary.sum():.1%}")

# Monthly cohort retention: activity in month n / cohort size
sales['InvoiceMonth'] = sales.InvoiceDate.dt.to_period('M')
first_month = sales.groupby('CustomerID').InvoiceMonth.min().rename('CohortMonth')
cohort_data = sales.join(first_month, on='CustomerID')
cohort_data['CohortIndex'] = ((cohort_data.InvoiceMonth.dt.year - cohort_data.CohortMonth.dt.year)*12 +
                               (cohort_data.InvoiceMonth.dt.month - cohort_data.CohortMonth.dt.month) + 1)
cohort_counts = cohort_data.groupby(['CohortMonth','CohortIndex']).CustomerID.nunique().unstack(fill_value=0)
cohort_retention = cohort_counts.div(cohort_counts.iloc[:,0], axis=0)
cohort_retention.iloc[:,:12].style.format('{:.0%}').background_gradient(cmap='Blues', vmin=0, vmax=1)

## 7. Segment action plan and measurement

Suggested actions are hypotheses, with a measurable outcome for each. Use randomized holdouts where possible; compare incremental margin and repeat purchase, not just opens or gross sales.


In [ ]:
action_plan = pd.DataFrame([
    ('Champions','Recognition, early access','Incremental 60-day repeat rate; margin per customer'),
    ('Loyal customers','Cross-sell or tiered rewards','Incremental items per order; margin after reward cost'),
    ('Potential loyalists','Relevant second-purchase recommendations','Second purchase within 45 days'),
    ('New customers','Welcome and product education','Second purchase within 45 days'),
    ('Promising','Low-cost nurture','Repeat purchase within 60 days'),
    ('Needs attention','Relevant reminder and product discovery','Reactivation within 60 days'),
    ('At risk','Personalized win-back test','Incremental reactivation and contribution margin'),
    ('Cannot lose them','High-touch win-back test','Incremental contribution margin vs holdout'),
    ('Hibernating','Low-cost reactivation or suppress','Positive incremental margin; unsubscribe rate'),
], columns=['Segment','Test idea','Primary measure'])
action_plan

## 8. Export analysis outputs

The exports include segment profiles, customer RFM assignments, cohort retention, and a campaign measurement starter table. Treat customer-level output as potentially sensitive business data.


In [ ]:
rfm.reset_index().to_csv('rfm_customers.csv', index=False)
summary.reset_index().to_csv('rfm_segments.csv', index=False)
cohort_retention.to_csv('rfm_cohorts.csv')
action_plan.to_csv('campaign_ideas.csv', index=False)
print('Exported rfm_customers.csv, rfm_segments.csv, rfm_cohorts.csv, and campaign_ideas.csv')